### Cài đặt môi trường

In [1]:
# Cài đặt các thư viện Web API và AI
%pip install fastapi uvicorn python-multipart nest-asyncio transformers torch pillow

!apt-get install -y ssh


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
]10;?]11;?PACMAN(8)                        Pacman Manual                        PACMAN(8)

NAME
     pacman - package manager utility

SYNOPSIS
     pacman <operation> [options] [targets]

DESCRIPTION
     Pacman is a package management utility that tracks installed packages on a
     Linux  system. It features dependency support, package groups, install and
     uninstall scripts, and the ability to sync your local machine with  a  re‐
     mote  repository  to automatically upgrade packages. Pacman packages are a
     zipped tar format.

     Since version 3.0.0, pacman has been  the  front-end  to  libalpm(3),  the
     “Arch  Linux  Package Management” library. This library allows alternative
     front-ends to be written (for instance, a GUI front-end).

     Invoking pacman involves specifying an opera

### Khởi tạo Server & Model

In [ ]:
%%writefile main.py
from fastapi import FastAPI, UploadFile, File, HTTPException
from transformers import pipeline
from PIL import Image
import io
import torch
import nest_asyncio
import matplotlib.pyplot as plt
from fastapi.staticfiles import StaticFiles
import os

# Tạo thư mục lưu kết quả trên Colab nếu chưa có
os.makedirs("outputs", exist_ok=True)

app = FastAPI()

depth_estimator = pipeline("depth-estimation", model="Intel/dpt-large")

@app.get("/")
def read_root():
    # Giới thiệu hệ thống
    return {
        "project": "2D-to-3D Depth Estimator API",
        "description": "API ước tính độ sâu từ ảnh 2D hỗ trợ làm game assets",
    }

@app.get("/health")
def health_check():
    # Kiểm tra trạng thái
    return {"status": "healthy", "model": "Intel/dpt-large", "device": "cuda" if torch.cuda.is_available() else "cpu"}

@app.post("/predict")
async def predict(file: UploadFile = File(None)):
    # Kiểm tra dữ liệu đầu vào và xử lý lỗi
    if file is None:
        raise HTTPException(status_code=400, detail="Lỗi: Vui lòng tải lên một file ảnh.")

    if not file.content_type.startswith('image/'):
        raise HTTPException(status_code=400, detail="Lỗi: Định dạng file không phải là hình ảnh.")

    try:
        # Đọc ảnh từ request
        contents = await file.read()
        image = Image.open(io.BytesIO(contents)).convert("RGB")

        # Suy luận (Inference)
        result = depth_estimator(image)
        depth_map = result["predicted_depth"]

        # Tạo tên file và lưu heatmap
        pure_filename = os.path.basename(file.filename)
        heatmap_filename = f"heatmap_{pure_filename}.png"
        heatmap_path = os.path.join("outputs", heatmap_filename)
    
        depth_numpy = depth_map.cpu().numpy()
        plt.imsave(heatmap_path, depth_numpy, cmap='magma')

        # Trả kết quả JSON
        return {
            "status": "success",
            "filename": file.filename,
            "heatmap_path": f"/outputs/{heatmap_filename}", # Đường dẫn tải heatmap
            "depth_map_info": {
                "width": depth_map.shape[1],
                "height": depth_map.shape[0],
                "max_depth": float(torch.max(depth_map)),
                "min_depth": float(torch.min(depth_map))
            }
        }
    except Exception as e:
        # Xử lý lỗi trong quá trình suy luận
        return {"status": "error", "message": str(e)}


Overwriting main.py


### Tạo Public URL (Pinggy Tunnel)

In [ ]:
import threading
import uvicorn
import nest_asyncio
import time
import subprocess
import re
import sys

nest_asyncio.apply()

# 1. Giải phóng cổng 8000 (Bắt buộc để không bị lỗi kết nối)
!fuser -k 8000/tcp

def run_app():
    # Chạy trên 0.0.0.0 để tunnel dễ nhận diện
    uvicorn.run("main:app", host="0.0.0.0", port=8000, reload=False)

# Khởi động server trong thread mới
new_thread = threading.Thread(target=run_app, name="run_app_thread", daemon=True)
new_thread.start()
time.sleep(5)

# 2. Khởi động Tunnel với cấu hình chuẩn
# Dùng free.pinggy.link thay vì qr@ để log sạch hơn, dễ bắt link hơn
proc = subprocess.Popen(
    ['ssh', '-o', 'StrictHostKeyChecking=no', '-p', '443',
     '-R', '80:localhost:8000', 'qr@a.pinggy.io'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    universal_newlines=True,
    bufsize=1
)

print(" Đang kết nối Pinggy...\n")
pinggy_url = None
output_buffer = ""

# Đọc từng ký tự để tránh bị kẹt do không có dấu xuống dòng
start_time = time.time()
while time.time() - start_time < 30: # Đợi tối đa 30s
    char = proc.stdout.read(1)
    if not char: break
    output_buffer += char
    
    # Regex mới: Linh hoạt hơn để bắt được cả định dạng .run.pinggy-free.link
    match = re.search(r'https://[a-z0-9-]+\.(run\.pinggy-free\.link|a\.free\.pinggy\.link)', output_buffer)
    if match:
        pinggy_url = match.group(0)
        print(f"{'='*70}")
        print(f"  LINK TÌM THẤY: {pinggy_url}")
        print(f"  Swagger UI: {pinggy_url}/docs")
        print(f"{'='*70}\n")
        break

if not pinggy_url:
    print("\n  Không tìm thấy link. Log thu được:")
    print(output_buffer)

/home/mieai/Documents/API/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


 Đang kết nối Pinggy...

Pseudo-terminal will not be allocated because stdin is not a terminal.


Exception in thread run_app_thread:
Traceback (most recent call last):
  File "/usr/lib/python3.14/threading.py", line 1082, in _bootstrap_inner
    self._context.run(self.run)
    ~~~~~~~~~~~~~~~~~^^^^^^^^^^
  File "/usr/lib/python3.14/threading.py", line 1024, in run
    self._target(*self._args, **self._kwargs)
    ~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/tmp/ipykernel_4389/915883161.py", line 9, in run_app
    uvicorn.run("main:app", host="127.0.0.1", port=8000, reload=False)
    ~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/home/mieai/Documents/API/.venv/lib/python3.14/site-packages/uvicorn/main.py", line 606, in run
    server.run()
    ~~~~~~~~~~^^
  File "/home/mieai/Documents/API/.venv/lib/python3.14/site-packages/uvicorn/server.py", line 75, in run
    return asyncio_run(self.serve(sockets=sockets), loop_factory=self.config.get_loop_factory())
  File "/usr/lib/python3.14/asyncio/runners.py", line 204, in run
    return runner.run(main)


** WARNING: connection is not using a post-quantum key exchange algorithm.
** This session may be vulnerable to "store now, decrypt later" attacks.
** The server may need to be upgraded. See https://openssh.com/pq.html
Allocated port 3 for remote forward to localhost:8000
You are not authenticated.
Your tunnel will expire in 60 minutes. Upgrade to Pinggy Pro to get unrestricted tunnels. https://dashboard.pinggy.io
http://sdncm-14-169-79-46.run.pinggy-free.link
https://sdncm-14-169-79-46.run.pinggy-free.link
RB: 0, SB: 0, TC: 0, AC: 4294967295
RB: 0, SB: 0, TC: 1, AC: 0
RB: 0, SB: 0, TC: 1, AC: 4294967295
RB: 0, SB: 0, TC: 2, AC: 0               connect_to localhost port 8000: failed.
RB: 0, SB: 0, TC: 2, AC: 0
RB: 0, SB: 0, TC: 2, AC: 4294967295
RB: 0, SB: 0, TC: 3, AC: 0
RB: 0, SB: 0, TC: 3, AC: 0               connect_to localhost port 8000: failed.
RB: 0, SB: 0, TC: 3, AC: 0
RB: 0, SB: 0, TC: 3, AC: 4294967295
RB: 0, SB: 0, TC: 4, AC: 0
RB: 0, SB: 0, TC: 4, AC: 0               conne